1. Importing packages and creating folders

In [1]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

raw_dir = Path("data/subset_raw")
clean_dir = Path("data/clean")
report_dir = Path("data/fastp_reports")
reference_dir = Path("data/reference")
quant_dir = Path("data/quant")
results_dir = Path("results")

for folder in [
    raw_dir,
    clean_dir,
    report_dir,
    reference_dir,
    quant_dir,
    results_dir,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders are ready.")

Project folders are ready.


2. Retrieving the metadata from the 12 samples

In [2]:
ena_url = (
    "https://www.ebi.ac.uk/ena/portal/api/filereport"
    "?accession=PRJNA1194989"
    "&result=read_run"
    "&fields=run_accession,sample_accession,experiment_title,library_layout"
    "&format=tsv"
)

samples = pd.read_csv(ena_url, sep="\t")

samples["sample_name"] = samples["experiment_title"].str.extract(
    r"GSM\d+:\s+(\S+)"
)

samples["condition"] = (
    samples["sample_name"]
    .str.replace(r"_\d+$", "", regex=True)
    .str.replace("624Mel_", "", regex=False)
)

samples["replicate"] = (
    samples["sample_name"]
    .str.extract(r"_(\d+)$")[0]
    .astype(int)
)

samples = samples.sort_values(
    ["condition", "replicate"]
).reset_index(drop=True)

display(
    samples[
        [
            "run_accession",
            "sample_name",
            "condition",
            "replicate",
            "library_layout",
        ]
    ]
)

print("Number of samples:", len(samples))

,run_accession,sample_name,condition,replicate,library_layout
0,SRR31631262,624Mel_siCTRL_1,siCTRL,1,PAIRED
1,SRR31631258,624Mel_siCTRL_2,siCTRL,2,PAIRED
2,SRR31631254,624Mel_siCTRL_3,siCTRL,3,PAIRED
3,SRR31631261,624Mel_siCTRL+IFNg_1,siCTRL+IFNg,1,PAIRED
4,SRR31631257,624Mel_siCTRL+IFNg_2,siCTRL+IFNg,2,PAIRED
5,SRR31631253,624Mel_siCTRL+IFNg_3,siCTRL+IFNg,3,PAIRED
6,SRR31631260,624Mel_siMITF_1,siMITF,1,PAIRED
7,SRR31631256,624Mel_siMITF_2,siMITF,2,PAIRED
8,SRR31631252,624Mel_siMITF_3,siMITF,3,PAIRED
9,SRR31631259,624Mel_siMITF+IFNg_1,siMITF+IFNg,1,PAIRED


Number of samples: 12


3. Downloading 1000000 reads from all 12 samples

In [3]:
max_spots = 1_000_000

for run in samples["run_accession"]:
    read1 = raw_dir / f"{run}_1.fastq.gz"
    read2 = raw_dir / f"{run}_2.fastq.gz"

    if read1.exists() and read2.exists():
        print(f"{run}: files already exist — skipping")
        continue

    print(f"Retrieving {max_spots:,} paired spots from {run}...")

    subprocess.run(
        [
            "fastq-dump",
            "-N", "1",
            "-X", str(max_spots),
            "--split-files",
            "--gzip",
            "--outdir", str(raw_dir),
            run,
        ],
        check=True,
    )

print("Subset retrieval complete.")

SRR31631262: files already exist — skipping
SRR31631258: files already exist — skipping
SRR31631254: files already exist — skipping
SRR31631261: files already exist — skipping
SRR31631257: files already exist — skipping
SRR31631253: files already exist — skipping
SRR31631260: files already exist — skipping
SRR31631256: files already exist — skipping
SRR31631252: files already exist — skipping
SRR31631259: files already exist — skipping
SRR31631255: files already exist — skipping
SRR31631251: files already exist — skipping
Subset retrieval complete.


4. Confirming the raw FASTQs; 24 in total, one reverse and one forward FASTQ from each sample

In [4]:
raw_fastqs = sorted(raw_dir.glob("*.fastq.gz"))

print("Raw FASTQ files:", len(raw_fastqs))

assert len(samples) == 12, "Expected 12 samples."
assert len(raw_fastqs) == 24, "Expected 24 paired-end FASTQ files."

for run in samples["run_accession"]:
    assert (raw_dir / f"{run}_1.fastq.gz").exists()
    assert (raw_dir / f"{run}_2.fastq.gz").exists()

print("All 12 paired-end samples are present.")

Raw FASTQ files: 24
All 12 paired-end samples are present.


5. Using FASTP to clean pair end reads; resulting in clean FASTQs and QC reports.

In [5]:
for run in samples["run_accession"]:
    input_r1 = raw_dir / f"{run}_1.fastq.gz"
    input_r2 = raw_dir / f"{run}_2.fastq.gz"

    output_r1 = clean_dir / f"{run}_1.clean.fastq.gz"
    output_r2 = clean_dir / f"{run}_2.clean.fastq.gz"

    json_report = report_dir / f"{run}.fastp.json"
    html_report = report_dir / f"{run}.fastp.html"

    if (
        output_r1.exists()
        and output_r2.exists()
        and json_report.exists()
        and html_report.exists()
    ):
        print(f"{run}: already cleaned — skipping")
        continue

    print(f"Cleaning {run}...")

    subprocess.run(
        [
            "fastp",
            "--in1", str(input_r1),
            "--in2", str(input_r2),
            "--out1", str(output_r1),
            "--out2", str(output_r2),
            "--json", str(json_report),
            "--html", str(html_report),
            "--detect_adapter_for_pe",
            "--thread", "4",
        ],
        check=True,
    )

print("Cleaning complete.")

Cleaning SRR31631262...


Detecting adapter sequence for read1...
>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123312436(98.6499%)
Q30 bases: 120357370(96.2859%)
Q40 bases: 0(0%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 122890277(98.3122%)
Q30 bases: 119404394(95.5235%)
Q40 bases: 0(0%)

Read1 after filtering:
total reads: 991434
total bases: 123501468
Q20 bases: 122078754(98.848%)
Q30 bases: 119240842(96.5501%)
Q40 bases: 0(0%)

Read2 after filtering:
total reads: 991434
total bases: 123502416
Q20 bases: 121939685(98.7347%)
Q30 bases: 118621314(96.0478%)
Q40 bases: 0(0%)

Filtering result:
reads passed filter: 1982868
reads failed due to low quality: 16638
reads failed due to too many N: 0
reads failed due to too short: 494
reads with adapter trimmed: 49269
bases trim

Cleaning SRR31631258...


Detecting adapter sequence for read1...
>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123368927(98.6951%)
Q30 bases: 120583393(96.4667%)
Q40 bases: 0(0%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 122622187(98.0977%)
Q30 bases: 119036618(95.2293%)
Q40 bases: 0(0%)

Read1 after filtering:
total reads: 989461
total bases: 123204709
Q20 bases: 121863759(98.9116%)
Q30 bases: 119206283(96.7546%)
Q40 bases: 0(0%)

Read2 after filtering:
total reads: 989461
total bases: 123207279
Q20 bases: 121495897(98.611%)
Q30 bases: 118118477(95.8697%)
Q40 bases: 0(0%)

Filtering result:
reads passed filter: 1978922
reads failed due to low quality: 20668
reads failed due to too many N: 0
reads failed due to too short: 410
reads with adapter trimmed: 53083
bases trim

Cleaning SRR31631254...


...

Cleaning SRR31631255...


>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123202768(98.5622%)
Q30 bases: 120300704(96.2406%)
Q40 bases: 0(0%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 122874726(98.2998%)
Q30 bases: 119508695(95.607%)
Q40 bases: 0(0%)

Read1 after filtering:
total reads: 990780
total bases: 123379104
Q20 bases: 121898298(98.7998%)
Q30 bases: 119129175(96.5554%)
Q40 bases: 0(0%)

Read2 after filtering:
total reads: 990780
total bases: 123381293
Q20 bases: 121813901(98.7296%)
Q30 bases: 118623801(96.1441%)
Q40 bases: 0(0%)

Filtering result:
reads passed filter: 1981560
reads failed due to low quality: 17976
reads failed due to too many N: 0
reads failed due to too short: 464
reads with adapter trimmed: 50881
bases trimmed due to adapters: 984848

Duplication

Cleaning SRR31631251...


>Illumina TruSeq Adapter Read 1
AGATCGGAAGAGCACACGTCTGAACTCCAGTCA

Detecting adapter sequence for read2...
>Illumina TruSeq Adapter Read 2
AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT

Read1 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123267776(98.6142%)
Q30 bases: 120327878(96.2623%)
Q40 bases: 0(0%)

Read2 before filtering:
total reads: 1000000
total bases: 125000000
Q20 bases: 123072781(98.4582%)
Q30 bases: 119808620(95.8469%)
Q40 bases: 0(0%)

Read1 after filtering:
total reads: 992142
total bases: 123620753
Q20 bases: 122185273(98.8388%)
Q30 bases: 119365410(96.5577%)
Q40 bases: 0(0%)

Read2 after filtering:
total reads: 992142
total bases: 123622675
Q20 bases: 122159568(98.8165%)
Q30 bases: 119045405(96.2974%)
Q40 bases: 0(0%)

Filtering result:
reads passed filter: 1984284
reads failed due to low quality: 15312
reads failed due to too many N: 0
reads failed due to too short: 404
reads with adapter trimmed: 40463
bases trimmed due to adapters: 840674

Duplicatio

Cleaning complete.
